# 04-statement-metric-tuning
## Ответ: Оптимальное p: 1.0; Лучшее качество: -16.03

## 1. Загрузите выборку Boston с помощьюфункции sklearn.datasets.load_boston(). Результатом вызова данной функции является объект, у которого признаки записаны в поле data, а целевой вектор в поле target.

### В новых версиях scikit-learn (>= 1.2) датасет Boston удвлен, поэтому загрузка вручную

In [4]:
import pandas as pd
import numpy as np

url_boston = "http://lib.stat.cmu.edu/datasets/boston"
boston_data = pd.read_csv(url_boston, sep=r"\s+", skiprows=22, header=None)
all_values = boston_data.values

X_features = np.hstack([all_values[::2, :], all_values[1::2, :2]])
y_target = all_values[1::2, 2]


# 2. Приведите признаки в выборке к одному масштабу при помощи функции sklearn.preprocessing.scale

In [5]:
from sklearn.preprocessing import scale
X_scaled = scale(X_features)

# 3. Переберите разные варианты параметра метрики p по сетке от 1 до 10 с таким шагом, чтобы всего было протестировано 200 вариантов (используйте функцию numpy.linspace). Используйте KNeighborsRegressor с n_neighbors=5 и weights=’distance’ данный параметр добавляет в алгоритм веса, зависящие от расстояния до ближайших соседей. В качестве метрики качества используйте среднеквадратичную ошиб ку (параметр scoring=’mean_squared_error’ у cross_val_score; при использовании библиотеки scikit-learn версии 18.0.1 и выше необхо димо указывать scoring=’neg_mean_squared_error’). Качество оце нивайте, как и в предыдущем задании, с помощью кросс-валидации по 5 блокам с random_state = 42, не забудьте включить перемеши вание выборки (shuffle=True).
# 4. Определите, при каком p качество на кросс-валидации оказалось оптимальным. Обратите внимание, что cross_val_score возвращает массив показателей качества по блокам; необходимо максимизировать среднее этих показателей. Это значение параметра и будет ответом на задачу

In [6]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, KFold

cv_folds = KFold(n_splits=5, shuffle=True, random_state=42)


p_values = np.linspace(1, 10, 200)

best_p = 1
best_score = -float('inf')

for current_p in p_values:
    knn_model = KNeighborsRegressor(
        n_neighbors=5, 
        weights='distance', 
        metric='minkowski', 
        p=current_p
    )
    
    cv_scores = cross_val_score(knn_model, X_scaled, y_target, cv=cv_folds, scoring='neg_mean_squared_error')
    avg_score = cv_scores.mean()
    
    if avg_score > best_score:
        best_score = avg_score
        best_p = current_p

print(f"Оптимальное p: {best_p}")
print(f"Лучшее качество: {best_score:.2f}")

Оптимальное p: 1.0
Лучшее качество: -16.03
